In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoTokenizer

In [ ]:
import os
from transformers import AutoTokenizer

# 定义训练好的模型的保存路径和模型名称
TRAINED_MODEL_PATH = "data/Qwen-GRPO-training"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 创建保存路径（如果不存在）
os.makedirs(TRAINED_MODEL_PATH, exist_ok=True)

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right"
)

# 如果分词器的 pad_token 没有被设置，则将其设置为 eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 保存分词器到指定路径
tokenizer.save_pretrained(TRAINED_MODEL_PATH)


In [ ]:
# 测试训练好的模型的推理能力
def test_trained_model_inference(user_input: str):
    """使用加载的训练好的模型和分词器进行推理测试。"""
    # 构建对话消息列表
    messages = [
        # 系统提示信息，复用之前定义的系统提示
        {"role": "system", "content": SYSTEM_PROMPT}, 
        # 用户输入的内容
        {"role": "user", "content": user_input}
    ]

    # 使用分词器应用聊天模板
    # 将消息列表转换为符合模型输入要求的文本格式
    # tokenize=False 表示不进行分词，仅生成文本
    # add_generation_prompt=True 表示添加生成提示
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 对输入文本进行分词
    # return_tensors="pt" 表示返回 PyTorch 张量
    # .to(device) 将分词后的输入数据移动到指定设备（如 GPU）上
    inputs = tokenizer(text, return_tensors="pt").to(device)

    # 使用训练好的模型生成输出
    outputs = trained_model.generate(
        # 解包输入参数
        **inputs,
        # 最大新生成的标记数量，这里设置为 200，可能会生成稍长的文本
        max_new_tokens=200, 
        # 是否进行采样，设置为 True 表示使用采样策略
        do_sample=True,
        # 采样温度，控制生成文本的随机性，值越大越随机
        temperature=0.7
    )

    # 将生成的标记解码为文本
    # skip_special_tokens=True 表示跳过特殊标记，只保留实际的文本内容
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 返回生成的文本响应
    return response